# Analytical SQL – Power Queries in Action
## Using CTE and Window Function

In [2]:
import sqlite3
import pandas as pd
conn=sqlite3.connect(r"C:\Users\shiva\Downloads\E Commerce data for sql\olist.sqlite")

In [3]:
#Nested Subquery
#Write a SQL query to find all products priced above the average price of their own product category.

query="""
select oi.product_id , p.product_category_name , oi.price
from order_items oi
join products p
    on p.product_id=oi.product_id
where oi.price >  (
    select avg(oi2.price)
from order_items oi2
join products p2
    on oi2.product_id=p2.product_id
where p2.product_category_name=p.product_category_name
    )
    limit 20

"""
df=pd.read_sql_query(query,conn)
df

,product_id,product_category_name,price
0,e5f2d52b802189ee658865ca93d83a8f,pet_shop,239.90
1,c777355d18b72b67abbeef9df44fd0fd,moveis_decoracao,199.00
2,ac6c3623068f30de03045865e4e10089,ferramentas_jardim,199.90
3,557d850972a7d6f792fd18ae1400d9b6,ferramentas_jardim,810.00
4,310ae3c140ff94b03219ad0adc3c778f,beleza_saude,145.95
5,3f27ac8e699df3d300ec4a5d8c5cf0b2,consoles_games,639.00
6,4fa33915031a8cde03dd0d3e8fb27f01,perfumaria,144.00
7,21b1c2f67a9aafb5af0eb06c13b9dbda,esporte_lazer,219.90
8,c389f712c4b4510bc997cee93e8b1a28,malas_acessorios,289.00
9,1c0c0093a48f13ba70d0c6b0a9157cb7,moveis_decoracao,109.90


In [4]:
#Find sellers whose total revenue is above the average revenue of all sellers.
#Identify top-performing sellers
#Benchmark individual seller performance
#Spot underperformers for audits

query = """
SELECT sum(price) as total_revenue , seller_id
FROM order_items
    group by seller_id
having sum(price) > (
select avg(total_revenue)
    from (
     select seller_id, sum(price) as total_revenue
     from order_items
     group by seller_id
    ) sub
)
"""
df = pd.read_sql_query(query, conn)
df

,total_revenue,seller_id
0,25080.03,001cca7ae9ae17fb1caed9dfb1094831
1,19712.71,004c9cd9d87a3c30c522c48c4fc07416
2,20260.00,00ee68308b45bc5e2660cd833c3f81cc
3,12684.90,00fc707aaaad2d31347cf883cd2dfe10
4,7058.00,014c0679dd340a0e338872e7ec85666a
...,...,...
623,12076.50,ff063b022a9a0aab91bad2c9088760b7
624,6771.00,ff1fb4c404b2efe68b03350a8dc24122
625,7017.00,ff4ea69c2a729e83e63c7579e4ef8170
626,21940.80,ff69aa92bb6b1bf9b8b7a51c2ed9cf8b


In [7]:
#Which customers had an average delivery time slower than the overall average delivery time ?
#Why this matters:
#Helps identify regions with logistics issues
#Can optimize delivery routes
#Customer satisfaction insights



query="""
select customer_id,
    round(avg(JULIANDAY(order_delivered_customer_date)-JULIANDAY(order_purchase_timestamp)),2) as avg_delivery_time
from orders
where order_delivered_customer_date is not null
group by  customer_id
having avg(JULIANDAY(order_delivered_customer_date)-JULIANDAY(order_purchase_timestamp)) > (
    select avg(JULIANDAY(order_delivered_customer_date)-JULIANDAY(order_purchase_timestamp))
    from orders
    where order_delivered_customer_date is not null
    )
"""
df=pd.read_sql_query(query,conn)
df


,customer_id,avg_delivery_time
0,00012a2ce6f8dcda20d059ce98491703,13.98
1,0002414f95344307404f0ace7a26f1d5,28.29
2,000419c5494106c306a97b5635748086,45.98
3,00050bf6e01e69d5c0fd612f1bcfb69c,15.22
4,00066ccbe787a588c52bd5ff404590e3,15.26
...,...,...
36375,fff55ba8dffa552b6fdfd86c2e806459,14.12
36376,fff5dd22d522cf28a902185817642a2e,19.68
36377,fff675a0d5924b9162b4a1bf410466cd,14.46
36378,fff89c8ed4fcf69a823c1d149e429a0b,18.10
